# Cello Sample Extractor

Extracts and classifies individual note samples from trimmed cello recordings.
Audio files are read from — and output written to — Google Drive.

**Before running:** `Runtime → Change runtime type → T4 GPU`

In [ ]:
# Verify GPU is available
import subprocess
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                   capture_output=True, text=True)
if r.returncode == 0:
    print('GPU:', r.stdout.strip())
else:
    print('WARNING: no GPU detected — go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
# Install dependencies
# Colab ships with tensorflow and numpy; we add the audio/signal-processing stack.
pip install -q soundfile scipy resampy tqdm
pip install -q crepe

In [ ]:
%%bash
# Clone the cello-sampler repo (skipped if already present)
if [ ! -d /content/cello ]; then
  git clone --quiet https://github.com/davidderoure/cello.git /content/cello
  echo "Cloned."
else
  git -C /content/cello pull --quiet
  echo "Updated."
fi

In [ ]:
import sys
sys.path.insert(0, '/content/cello')

# Verify the package is importable
import cello_sampler
print('cello_sampler loaded from', cello_sampler.__file__)

## Configure paths

Edit the paths below to match your Google Drive layout.

In [ ]:
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive')

# ── Input files (trimmed, silence/talking removed) ──────────────────────────
TRIMMED = DRIVE / 'cello_session' / 'trimmed'

INPUT_FILES = {
    'mono_1': TRIMMED / 'mono_1.wav',
    'mono_2': TRIMMED / 'mono_2.wav',
    'stereo': TRIMMED / 'stereo.wav',
}

# ── Output root (sub-directories created per mic) ───────────────────────────
OUTPUT_ROOT = DRIVE / 'cello_session' / 'samples'

# Verify inputs exist
for name, path in INPUT_FILES.items():
    status = '✓' if path.exists() else '✗ NOT FOUND'
    print(f'  {name}: {path}  {status}')

## Optional: override detection thresholds

Leave unchanged to use the defaults.  See the README for guidance on tuning.

In [ ]:
from cello_sampler import config

# Onset detection
config.ONSET_THRESHOLD_MULTIPLIER = 1.5   # raise if too many false onsets
config.MIN_NOTE_DURATION_MS       = 50.0

# Quality gating
config.PITCH_CONFIDENCE_THRESHOLD    = 0.85  # lower if too many low-confidence rejections
config.MAX_INTONATION_DEVIATION_CENTS = 30.0
config.POLYPHONY_SALIENCE_THRESHOLD   = 0.30  # raise if clean notes are wrongly rejected

# Articulation
config.PIZZICATO_MAX_ATTACK_MS      = 15.0
config.STACCATO_MAX_DURATION_MS     = 250.0
config.VIBRATO_MIN_DEPTH_CENTS      = 20.0

# Workers: keep at 1 in Colab — each extra process re-loads TensorFlow
# and fights for the same GPU, which is slower not faster.
N_WORKERS = 1

print('Thresholds configured.')

## Run the pipeline

In [ ]:
from cello_sampler.pipeline import run

results = {}

for mic_name, input_path in INPUT_FILES.items():
    if not input_path.exists():
        print(f'Skipping {mic_name}: file not found at {input_path}')
        continue

    output_dir = OUTPUT_ROOT / mic_name
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f'\n── {mic_name} ──────────────────────────────────')
    print(f'   Input  : {input_path}')
    print(f'   Output : {output_dir}')

    result = run(
        input_path=input_path,
        output_dir=output_dir,
        n_workers=N_WORKERS,
    )

    results[mic_name] = result
    print(f'   Accepted : {result.n_accepted}')
    print(f'   Rejected : {result.n_rejected}')

print('\nDone.')

## Summary

In [ ]:
import csv
from collections import Counter

for mic_name, result in results.items():
    index_path = OUTPUT_ROOT / mic_name / '_index.csv'
    if not index_path.exists():
        continue

    with index_path.open() as f:
        rows = list(csv.DictReader(f))

    articulations = Counter(r['articulation'] for r in rows)
    notes         = Counter(r['note'] for r in rows)

    print(f'\n{mic_name}')
    print(f'  Total accepted : {result.n_accepted}')
    print(f'  Total rejected : {result.n_rejected}')
    print(f'  By articulation: {dict(articulations)}')
    print(f'  Unique pitches : {len(notes)}  ({" ".join(sorted(notes))})')